# exp_013 — NEW-METHODS sweep (Colab, crash-safe incremental zips)

The calibration zip has **only icm + rnd** (4 games × 3 levels × 8 seeds). We have **no A/B/C data**.
This runs our new methods and **saves a zip after every single run** so a Colab disconnect never
loses more than the run in flight.

| method | what | module |
|---|---|---|
| **A** `frozen-φ` | RND+leak on a frozen-random φ (no ICM) — most robust where ICM-φ fails | `exp_013_1 --phi-mode frozen` |
| **B** `rnd_icm` | RND+leak on ICM φ | `exp_013_1` |
| **C** `additive` | ½·ICM + ½·RND-on-φ | `exp_013_2` |

**Plan:** Tier 1 = A/B/C on **L1 of all 4 games** (places them next to icm/rnd/random on the main
figure). Tier 2 = A/B/C on the **frontier** (ls20/g50t/re86 L2+L3, tu93 L3 — where icm/rnd scored 0/8).
Tier 1 runs first so the cheap, high-value comparison data lands early.

> D (lookahead) is omitted by default — its Q is anti-informative (see `probes/method_improvements.md`).
> C will be weak on long frontier runs until its normalizer self-suppression is fixed (also noted there).

> **Set Runtime ▸ GPU.**

## 1. Setup

In [ ]:
import os, sys, glob, json, time
REPO_URL = "https://github.com/LavetteSinsora/ProjectArceus.git"; REPO = "/content/ProjectArceus"
if not os.path.isdir(REPO):
    !git clone --depth 1 $REPO_URL $REPO
%cd /content/ProjectArceus
!git pull --ff-only -q || true
!pip -q install "arc-agi>=0.9.8" "arcengine>=0.9.3"
!pip -q install -e . --no-deps
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(set Runtime→GPU!)")

## 2. Persistence — where the incremental zip lives

**Google Drive is recommended** for `--zip-each`: it survives a Colab disconnect, so the latest
`exp013_progress.zip` always holds every completed run. (Per-run `files.download` can't fire
reliably in the background, which is why we persist instead.) To skip Drive, set
`SAVE_DIR='/content/exp013_out'` and download the zip manually from the Files pane.

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/exp013_new_methods"
else:
    SAVE_DIR = "/content/exp013_out"
os.makedirs(SAVE_DIR, exist_ok=True)
print("incremental zips ->", SAVE_DIR, "(file: exp013_progress.zip, overwritten after each run)")

## 3. Config (edit for your budget)

In [ ]:
METHODS         = ["A", "B", "C"]     # D (lookahead) intentionally EXCLUDED; E never validated
SEEDS_L1        = [0, 1, 2, 3]         # Tier 1 (comparison) — want decent statistics
SEEDS_FRONTIER  = [0, 1]              # Tier 2 (frontier) — expensive, mostly censored
FRONTIER_CAP    = 400_000             # uniform cap for the L2/L3 frontier cells
CONCURRENCY     = 2                    # parallel runs (small models; 2-3 use a GPU's headroom)

SWEEP = "JEPA.experiments.exp_013_headline_experiment.sweep"
def args_to_str(a): return " ".join(map(str, a))
print("methods", METHODS, "| L1 seeds", SEEDS_L1, "| frontier seeds", SEEDS_FRONTIER, "| cap", FRONTIER_CAP)

## 4. Tier 1 — comparison (A/B/C on L1 of all 4 games)

Per-cell caps (ls20 200k · tu93 600k · re86 1M · g50t 300k). Each run updates `exp013_progress.zip`.

In [ ]:
T1 = args_to_str(["--games", "ls20", "tu93", "re86", "g50t", "--levels", 0,
                  "--methods", *METHODS, "--seeds", *SEEDS_L1,
                  "--no-transfer", "--concurrency", CONCURRENCY,
                  "--zip-each", "--save-dir", SAVE_DIR, "--logdir", "/content/logs_t1"])
print("Tier1:", T1)
!python -m $SWEEP $T1

## 5. Tier 2 — frontier (A/B/C where icm/rnd were 0/8)

ls20/g50t/re86 **L2+L3**, then tu93 **L3**. Mostly censored at `FRONTIER_CAP` — a single solve is the
headline result. Still incremental-zipped after each run.

In [ ]:
T2 = args_to_str(["--games", "ls20", "g50t", "re86", "--levels", 1, 2,
                  "--methods", *METHODS, "--seeds", *SEEDS_FRONTIER,
                  "--cap", FRONTIER_CAP, "--no-transfer", "--concurrency", CONCURRENCY,
                  "--zip-each", "--save-dir", SAVE_DIR, "--logdir", "/content/logs_t2"])
print("Tier2a:", T2)
!python -m $SWEEP $T2

T3 = args_to_str(["--games", "tu93", "--levels", 2,
                  "--methods", *METHODS, "--seeds", *SEEDS_FRONTIER,
                  "--cap", FRONTIER_CAP, "--no-transfer", "--concurrency", CONCURRENCY,
                  "--zip-each", "--save-dir", SAVE_DIR, "--logdir", "/content/logs_t3"])
print("Tier2b (tu93 L3):", T3)
!python -m $SWEEP $T3

## 6. Grab the latest results zip (also already in `SAVE_DIR` after every run)

In [ ]:
zp = os.path.join(SAVE_DIR, "exp013_progress.zip")
print("latest zip:", zp, f"({os.path.getsize(zp)/1e6:.1f} MB)" if os.path.exists(zp) else "(none yet)")
try:
    from google.colab import files; files.download(zp)
except Exception as e:
    print("Download from the Files pane / Drive:", zp, e)